# Model Evaluation

This notebook evaluates the saved model on data that was not used during training.

A good evaluation reports more than accuracy. We will inspect precision, recall, weighted F1, and a confusion matrix for each startup status.

## Evaluation target

The model is evaluated against the `status` column. This is the correct target because it represents the company outcome we want to predict.

The four classes are `operating`, `acquired`, `closed`, and `ipo`. We compare the model's predicted status with the real status and report metrics for each class.

In [ ]:
from pathlib import Path

import pickle
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


def find_project_root() -> Path:
    current_folder = Path.cwd().resolve()
    for folder in [current_folder, *current_folder.parents]:
        if (folder / "src" / "data" / "companies.csv").exists():
            return folder
    raise FileNotFoundError("Could not find src/data/companies.csv")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "src" / "data" / "companies.csv"
MODEL_PATH = PROJECT_ROOT / "src" / "models" / "best_model.pkl"
TARGET_COLUMN = "status"
valid_statuses = ["operating", "acquired", "closed", "ipo"]
columns_to_drop = [
    TARGET_COLUMN, "id", "Unnamed: 0.1", "entity_type", "entity_id", "parent_id",
    "name", "normalized_name", "permalink", "domain", "homepage_url",
    "twitter_username", "logo_url", "logo_width", "logo_height", "short_description",
    "description", "overview", "tag_list", "created_by", "created_at", "updated_at",
    "first_investment_at", "last_investment_at", "first_funding_at", "last_funding_at",
    "first_milestone_at", "last_milestone_at", "closed_at", "ROI",
]

data = pd.read_csv(DATA_PATH)
data[TARGET_COLUMN] = data[TARGET_COLUMN].astype("string").str.strip().str.lower()
data = data[data[TARGET_COLUMN].isin(valid_statuses)].drop_duplicates().reset_index(drop=True)
X = data[[column for column in data.columns if column not in columns_to_drop]]
y = data[TARGET_COLUMN]
_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

with MODEL_PATH.open("rb") as model_file:
    model_artifact = pickle.load(model_file)
model = model_artifact["pipeline"]
target_encoder = model_artifact["target_encoder"]
prediction_codes = model.predict(X_test)
predictions = target_encoder.inverse_transform(prediction_codes.astype(int))
print(f"Loaded {model_artifact['model_name']} from: {MODEL_PATH}")
print(f"Evaluation rows: {len(X_test):,}")

Loaded model from: /home/chandan/Work/ANN_RiceClassification_Using_Machine-Learning/src/models/best_model.pkl
Evaluation rows: 39,311


In [ ]:
print(classification_report(y_test, predictions, zero_division=0))

report = pd.DataFrame(
    classification_report(y_test, predictions, output_dict=True, zero_division=0)
).transpose()
report

In [ ]:
labels = sorted(y_test.unique())
confusion = confusion_matrix(y_test, predictions, labels=labels)

print("Rows are the true classes; columns are the predicted classes.")
print(pd.DataFrame(confusion, index=labels, columns=labels))

ConfusionMatrixDisplay(confusion_matrix=confusion, display_labels=labels).plot(
    xticks_rotation=45,
    cmap="Blues",
)
plt.title("Confusion matrix on the held-out test set")
plt.tight_layout()
plt.show()

## Evaluation checklist

Before using the model, check the per-class recall and F1 scores, not only the overall score. A low score for one status means the model needs improvement for that class, even when the total accuracy looks acceptable.

The evaluation uses the same held-out split and feature contract as training. This helps make the reported metrics reproducible and prevents post-outcome fields from entering the test data.

The next notebook shows the smallest prediction function needed by an application.